# 02b — Image Captioning en Colab (desde Google Drive)

Versión que **no requiere GitHub**. Subes la carpeta `proyectodeep/` a tu Drive y este notebook la usa.

**Pasos previos:**
1. Sube la carpeta `proyectodeep/` a Google Drive (Mi unidad/proyectodeep). No hace falta subir `Images/` ni `.venv/`.
2. Abre este notebook en Colab → *Runtime → Change runtime type → T4 GPU*.
3. Ejecuta de arriba a abajo.

## 1. Verificar GPU

In [1]:
!nvidia-smi

Mon Apr 27 21:14:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar Google Drive y entrar al proyecto

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Proyectodeep'
os.chdir(PROJECT_DIR)
print('Working in:', os.getcwd())
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/Proyectodeep
captions.txt  notebooks  README.md  requirements.txt  src


## 3. Instalar dependencias

In [8]:
!pip install -q wandb

## 4. Descargar Flickr8k desde Kaggle

Necesitas tu `kaggle.json` (https://www.kaggle.com/settings → *Create New Token*).
Las imágenes se descargan a Drive solo la primera vez (~1 GB, tarda 2-5 min).

In [14]:
import os
!unzip -oq data/flickr8k/flickr8k.zip -d /content/flickr8k_local
!rm -rf data/flickr8k/Images
!ln -s /content/flickr8k_local/Images data/flickr8k/Images
print('Imágenes:', len(os.listdir('data/flickr8k/Images')))

Imágenes: 8091


## 5. Construir vocabulario

In [ ]:
!python -m src.vocabulary --captions data/flickr8k/captions.txt --out data/flickr8k/vocab.pkl --threshold 5

## 6. Login en Weights & Biases

Te pedirá tu API key (https://wandb.ai/authorize).

In [15]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:


KeyboardInterrupt: 

In [18]:
!cp /content/flickr8k_local/captions.txt data/flickr8k/captions.txt
!ls data/flickr8k/

captions.txt  flickr8k.zip  Images


In [21]:
import os, pandas as pd
df = pd.read_csv('data/flickr8k/captions.txt')
existing = set(os.listdir('data/flickr8k/Images'))
before = len(df)
df = df[df['image'].isin(existing)]
print(f'Filas: {before} → {len(df)}  (descartadas {before-len(df)})')
df.to_csv('data/flickr8k/captions.txt', index=False)

Filas: 40455 → 39715  (descartadas 740)


In [22]:
!python -m src.vocabulary --captions data/flickr8k/captions.txt --out data/flickr8k/vocab.pkl --threshold 5

Vocab size: 2954 (threshold=5)
Saved to data/flickr8k/vocab.pkl


## 7. Entrenamiento

~5-7 min por época en T4. 5 épocas ≈ 30 min.

In [23]:
!python -m src.train \
    --epochs 5 \
    --batch-size 64 \
    --num-workers 2 \
    --lr 1e-3 \
    --backbone resnet50

[device] cuda
[vocab] size = 2954
[data] train batches=465  val batches=79
epoch 1/5  step 0/465  loss=7.9919  ppl=2956.89
epoch 1/5  step 20/465  loss=5.0149  ppl=150.64
epoch 1/5  step 40/465  loss=4.2626  ppl=70.99
epoch 1/5  step 60/465  loss=4.0301  ppl=56.27
epoch 1/5  step 80/465  loss=3.8157  ppl=45.41
epoch 1/5  step 100/465  loss=3.6729  ppl=39.36
epoch 1/5  step 120/465  loss=3.6339  ppl=37.86
epoch 1/5  step 140/465  loss=3.5231  ppl=33.89
epoch 1/5  step 160/465  loss=3.2520  ppl=25.84
epoch 1/5  step 180/465  loss=3.3431  ppl=28.31
epoch 1/5  step 200/465  loss=3.2732  ppl=26.39
epoch 1/5  step 220/465  loss=3.4997  ppl=33.11
epoch 1/5  step 240/465  loss=3.5281  ppl=34.06
epoch 1/5  step 260/465  loss=3.2177  ppl=24.97
epoch 1/5  step 280/465  loss=3.2335  ppl=25.37
epoch 1/5  step 300/465  loss=3.2106  ppl=24.80
epoch 1/5  step 320/465  loss=3.2080  ppl=24.73
epoch 1/5  step 340/465  loss=3.0347  ppl=20.80
epoch 1/5  step 360/465  loss=3.2734  ppl=26.40
epoch 1/5  step 

## 8. Generar captions de ejemplo

In [29]:
import sys, random, glob
sys.path.insert(0, '.')
from src.vocabulary import Vocabulary

import torch, matplotlib.pyplot as plt
from PIL import Image
from src.sample import load_checkpoint, caption_image
from src.dataset import split_image_ids

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = sorted(glob.glob('checkpoints/ckpt_epoch*.pt'))[-1]
encoder, decoder, vocab = load_checkpoint(ckpt, 'data/flickr8k/vocab.pkl', device)

# Reconstruir el split (mismo seed=42 que en training)
_, _, test_ids = split_image_ids('data/flickr8k/captions.txt', seed=42)
print(f'Imágenes de test disponibles: {len(test_ids)}')

samples = random.sample(list(test_ids), 6)
samples = [f'data/flickr8k/Images/{img}' for img in samples]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.flat, samples):
    cap = caption_image(path, encoder, decoder, vocab, device)
    ax.imshow(Image.open(path)); ax.axis('off')
    ax.set_title(cap, fontsize=10)
plt.suptitle('Captions on TEST images (unseen during training)', fontsize=14)
plt.tight_layout(); plt.savefig('captions_test.png', dpi=120); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [30]:
import sys, random, glob, pandas as pd
sys.path.insert(0, '.')
from src.vocabulary import Vocabulary

import torch, matplotlib.pyplot as plt
from PIL import Image
from src.sample import load_checkpoint, caption_image
from src.dataset import split_image_ids
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk; nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = sorted(glob.glob('checkpoints/ckpt_epoch*.pt'))[-1]
encoder, decoder, vocab = load_checkpoint(ckpt, 'data/flickr8k/vocab.pkl', device)

# Test split
_, _, test_ids = split_image_ids('data/flickr8k/captions.txt', seed=42)
df = pd.read_csv('data/flickr8k/captions.txt')

# Cogemos 8 imágenes random de test
samples = random.sample(list(test_ids), 8)
smooth = SmoothingFunction().method1

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
bleu_scores = []
for ax, img_id in zip(axes.flat, samples):
    path = f'data/flickr8k/Images/{img_id}'
    pred = caption_image(path, encoder, decoder, vocab, device)
    # Referencias reales (5 captions por imagen en Flickr8k)
    refs = df[df['image'] == img_id]['caption'].tolist()
    refs_tok = [r.lower().split() for r in refs]
    pred_tok = pred.lower().replace('<start>', '').replace('<end>', '').split()
    bleu = sentence_bleu(refs_tok, pred_tok, weights=(0.5, 0.5),
                          smoothing_function=smooth)
    bleu_scores.append(bleu)

    ax.imshow(Image.open(path)); ax.axis('off')
    title = f'PRED: {pred}\nREF:  {refs[0]}\nBLEU-2: {bleu:.3f}'
    ax.set_title(title, fontsize=9, loc='left')

plt.suptitle(f'Test images — Avg BLEU-2: {sum(bleu_scores)/len(bleu_scores):.3f}',
             fontsize=14)
plt.tight_layout()
plt.savefig('captions_with_bleu.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nMedia BLEU-2: {sum(bleu_scores)/len(bleu_scores):.3f}')
print(f'Min: {min(bleu_scores):.3f}  Max: {max(bleu_scores):.3f}')

Output hidden; open in https://colab.research.google.com to view.

## 9. (Opcional) Subir tabla de muestras a W&B

In [ ]:
import wandb
run = wandb.init(project='image-captioning', name='samples', job_type='inference')
table = wandb.Table(columns=['image', 'generated_caption'])
for path in samples:
    cap = caption_image(path, encoder, decoder, vocab, device)
    table.add_data(wandb.Image(path), cap)
run.log({'samples': table}); run.finish()